1.Which are the top 3 restaurants based on revenue generated from successfully delivered orders?

In [0]:
%sql
SELECT
    r.restaurant_name,
    COUNT(f.order_id) AS delivered_orders_count,
    ROUND(SUM(f.order_amount), 2) AS total_delivered_revenue
FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt f
LEFT JOIN food_delivery_catalog.gold.dim_restaurant r
    ON f.restaurant_sk = r.restaurant_sk
WHERE f.order_status = 'DELIVERED'
GROUP BY r.restaurant_name
ORDER BY total_delivered_revenue DESC
LIMIT 3;

Databricks visualization. Run in Databricks to view.

Weekly revenue trend Business question: How does delivered revenue change week by week, and which week generates the highest revenue?

Which cities generate the highest delivered-order revenue?

In [0]:
%sql
SELECT
    c.city_name,
    COUNT(f.order_id) AS delivered_orders_count,
    ROUND(SUM(f.order_amount), 2) AS total_delivered_revenue
FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt f
LEFT JOIN food_delivery_catalog.gold.dim_city c
    ON f.city_sk = c.city_sk
WHERE f.order_status = 'DELIVERED'
GROUP BY c.city_name
ORDER BY total_delivered_revenue DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH weekly_revenue AS (
    SELECT
        YEAR(d.order_date) AS order_year,
        WEEKOFYEAR(d.order_date) AS week_number,
        CAST(DATE_TRUNC('WEEK', d.order_date) AS DATE) AS week_start_date,
        SUM(f.order_amount) AS total_revenue
    FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt f
    JOIN food_delivery_catalog.dbt_work_gold.dim_date_dbt d
        ON f.date_sk = d.date_sk
    WHERE f.order_status = 'DELIVERED'
    GROUP BY
        YEAR(d.order_date),
        WEEKOFYEAR(d.order_date),
        CAST(DATE_TRUNC('WEEK', d.order_date) AS DATE)
),

revenue_comparison AS (
    SELECT
        *,
        LAG(total_revenue) OVER (
            PARTITION BY order_year
            ORDER BY week_number
        ) AS previous_week_revenue
    FROM weekly_revenue
)

SELECT
    order_year,
    week_number,
    week_start_date,
    ROUND(total_revenue, 2) AS current_week_revenue,
    ROUND(previous_week_revenue, 2) AS previous_week_revenue,
    ROUND(
        100.0 * (total_revenue - previous_week_revenue)
        / NULLIF(previous_week_revenue, 0),
        2
    ) AS revenue_growth_percentage
FROM revenue_comparison
ORDER BY order_year, week_number;

Databricks visualization. Run in Databricks to view.

Which restaurant generates the highest gross sales, and how much discount does it offer?

In [0]:
%sql
SELECT
    r.restaurant_name,
    SUM(f.quantity * f.item_price + f.delivery_fee) AS gross_sales,
    SUM(f.discount) AS revenue_lost_through_discounts,
    SUM(f.order_amount) AS net_revenue
FROM food_delivery_catalog.dbt_work_gold.fact_orders_dbt f
LEFT JOIN food_delivery_catalog.gold.dim_restaurant r
    ON f.restaurant_sk = r.restaurant_sk
WHERE f.order_status = 'DELIVERED'
GROUP BY r.restaurant_name
ORDER BY gross_sales DESC;


Databricks visualization. Run in Databricks to view.